# Capstone Project — Deploy a Multi-Service Application with a Custom Helm Chart

---

## Scenario

You are a platform engineer at **Taskly**, a task management startup. The development team has built a two-service application:

- **`taskly-api`** — a Python/FastAPI backend that serves REST endpoints
- **`taskly-frontend`** — an Nginx-based frontend that serves the SPA

Both services need a **PostgreSQL** database, and the frontend proxies to the backend.

Your job is to author a production-grade Helm chart that:

1. Deploys both services with configurable replicas, images, and resources
2. Bundles PostgreSQL as a sub-chart dependency
3. Supports dev, staging, and production environment overrides
4. Runs a `pre-upgrade` database migration hook
5. Includes `helm test` smoke tests
6. Passes `helm lint` cleanly
7. Can be packaged and pushed to an OCI registry

---

## What You'll Build

```
taskly/                        # Root chart
├── Chart.yaml                 # name: taskly, version: 1.0.0
├── values.yaml                # Base defaults
├── values-dev.yaml            # Dev overrides
├── values-staging.yaml        # Staging overrides
├── values-production.yaml     # Production overrides
├── values.schema.json         # Schema validation
├── Chart.lock
├── charts/
│   └── postgresql-xx.tgz      # Downloaded dependency
└── templates/
    ├── _helpers.tpl
    ├── NOTES.txt
    ├── api-deployment.yaml
    ├── api-service.yaml
    ├── frontend-deployment.yaml
    ├── frontend-service.yaml
    ├── ingress.yaml
    ├── configmap.yaml
    ├── hooks/
    │   └── pre-upgrade-migrate.yaml
    └── tests/
        ├── test-api-connection.yaml
        └── test-frontend-connection.yaml
```

---

## Prerequisites

Ensure you have completed:
- Lessons 1.1–1.4 (Helm CLI, install, upgrade)
- Lessons 2.1–2.4 (Chart structure, templates, built-in objects)
- Lessons 3.1–3.4 (Functions, control flow, helpers, schema)
- Lessons 4.1–4.3 (Dependencies, subcharts, hooks)
- Lessons 5.1–5.4 (OCI, testing, best practices, CI/CD)

---

## Part 1 — Scaffold the Chart

In [ ]:
# TODO: Create the chart scaffold
# 1. Create the directory structure at /tmp/taskly
# 2. Write Chart.yaml with name: taskly, version: 1.0.0, appVersion: "1.0.0"
# 3. Include a postgresql dependency in Chart.yaml (bitnami/postgresql, version 15.x.x)

# Reference: Lesson 2.1 (chart structure), Lesson 4.1 (dependencies)

# Starter structure:
rm -rf /tmp/taskly && mkdir -p /tmp/taskly/templates/hooks /tmp/taskly/templates/tests

# TODO: Write Chart.yaml
cat > /tmp/taskly/Chart.yaml << 'EOF'
apiVersion: v2
name: taskly
description: Taskly — a two-service task management application
type: application
version: 1.0.0
appVersion: "1.0.0"
# TODO: Add the postgresql dependency here
EOF

echo "Part 1 started — fill in Chart.yaml and run helm dependency update"

---

## Part 2 — Design values.yaml

Design a `values.yaml` that supports both services with independent configuration.

In [ ]:
%%writefile /tmp/taskly/values.yaml
# TODO: Complete this values.yaml

# Global values (shared by all subcharts)
global:
  imageRegistry: ""   # Set to override all image registries

# --- Taskly API (backend) ---
api:
  # TODO: Add replicaCount, image (repository/tag/pullPolicy), service, resources
  replicaCount: 1
  image:
    repository: python
    tag: "3.12-slim"   # Replace with your real API image in production
    pullPolicy: IfNotPresent
  service:
    type: ClusterIP
    port: 8000
  resources:
    limits:
      cpu: 500m
      memory: 256Mi
    requests:
      cpu: 100m
      memory: 128Mi

# --- Taskly Frontend ---
frontend:
  # TODO: Add replicaCount, image, service
  replicaCount: 1
  image:
    repository: nginx
    tag: "1.25"
    pullPolicy: IfNotPresent
  service:
    type: ClusterIP
    port: 80

# --- Ingress ---
ingress:
  enabled: false
  className: ""
  annotations: {}
  host: taskly.example.local
  tls: false

# --- Database migration ---
migration:
  enabled: true
  image:
    repository: alpine
    tag: "3.19"

# --- PostgreSQL subchart config ---
postgresql:
  enabled: true
  auth:
    database: taskly
    username: taskly
    # TODO: Use existingSecret in production. For local dev only:
    password: changemeplease
  primary:
    persistence:
      size: 1Gi

---

## Part 3 — Write the Helpers

In [ ]:
# TODO: Write _helpers.tpl with these named templates:
#   - taskly.fullname           (Release.Name + chart name, truncated)
#   - taskly.labels             (standard k8s labels)
#   - taskly.selectorLabels     (stable selector labels)
#   - taskly.apiFullname        (fullname + "-api")
#   - taskly.frontendFullname   (fullname + "-frontend")
#   - taskly.dbHost             (postgresql service host)

# Reference: Lesson 3.3 (helpers and partials)

cat > /tmp/taskly/templates/_helpers.tpl << 'EOF'
{{/*
  taskly.fullname — base name for resources
*/}}
{{- define "taskly.fullname" -}}
{{- printf "%s" .Release.Name | trunc 63 | trimSuffix "-" }}
{{- end }}

{{/*
  taskly.apiFullname
*/}}
{{- define "taskly.apiFullname" -}}
{{- printf "%s-api" (include "taskly.fullname" .) | trunc 63 | trimSuffix "-" }}
{{- end }}

{{/*
  taskly.frontendFullname
*/}}
{{- define "taskly.frontendFullname" -}}
{{- printf "%s-frontend" (include "taskly.fullname" .) | trunc 63 | trimSuffix "-" }}
{{- end }}

{{/*
  taskly.labels — standard Kubernetes recommended labels
*/}}
{{- define "taskly.labels" -}}
helm.sh/chart: {{ printf "%s-%s" .Chart.Name .Chart.Version | replace "+" "_" | trunc 63 | trimSuffix "-" }}
app.kubernetes.io/name: {{ .Chart.Name }}
app.kubernetes.io/instance: {{ .Release.Name }}
app.kubernetes.io/version: {{ .Chart.AppVersion | quote }}
app.kubernetes.io/managed-by: {{ .Release.Service }}
{{- end }}

{{/*
  taskly.selectorLabels — stable labels safe for use in selectors
*/}}
{{- define "taskly.selectorLabels" -}}
app.kubernetes.io/name: {{ .Chart.Name }}
app.kubernetes.io/instance: {{ .Release.Name }}
{{- end }}

{{/*
  taskly.dbHost — the hostname of the postgresql service
*/}}
{{- define "taskly.dbHost" -}}
{{- printf "%s-postgresql" .Release.Name }}
{{- end }}
EOF

echo "_helpers.tpl written"

---

## Part 4 — Write the API Deployment & Service

In [ ]:
# TODO: Write api-deployment.yaml using .Values.api.* values
# Must use:
#   - taskly.apiFullname helper for the name
#   - taskly.labels helper for labels
#   - taskly.selectorLabels helper for selector
#   - DB connection env vars: DATABASE_URL using taskly.dbHost helper
#   - Resource limits from .Values.api.resources

# Reference: Lesson 2.3 (writing templates), Lesson 3.3 (helpers)

cat > /tmp/taskly/templates/api-deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "taskly.apiFullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
    app.kubernetes.io/component: api
spec:
  replicas: {{ .Values.api.replicaCount }}
  selector:
    matchLabels:
      {{- include "taskly.selectorLabels" . | nindent 6 }}
      app.kubernetes.io/component: api
  template:
    metadata:
      labels:
        {{- include "taskly.selectorLabels" . | nindent 8 }}
        app.kubernetes.io/component: api
    spec:
      containers:
        - name: api
          image: {{ .Values.api.image.repository }}:{{ .Values.api.image.tag }}
          imagePullPolicy: {{ .Values.api.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.api.service.port }}
          env:
            - name: DATABASE_URL
              value: {{ printf "postgresql://%s:%s@%s:5432/%s" .Values.postgresql.auth.username .Values.postgresql.auth.password (include "taskly.dbHost" .) .Values.postgresql.auth.database | quote }}
            - name: APP_ENV
              value: {{ .Values.global.environment | default "development" | quote }}
          # TODO: Add readinessProbe and livenessProbe
          resources:
            {{- toYaml .Values.api.resources | nindent 12 }}
EOF

cat > /tmp/taskly/templates/api-service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: {{ include "taskly.apiFullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
    app.kubernetes.io/component: api
spec:
  type: {{ .Values.api.service.type }}
  ports:
    - port: {{ .Values.api.service.port }}
      targetPort: {{ .Values.api.service.port }}
      protocol: TCP
  selector:
    {{- include "taskly.selectorLabels" . | nindent 4 }}
    app.kubernetes.io/component: api
EOF

echo "API templates written"

---

## Part 5 — Write the Frontend Deployment & Service

In [ ]:
# TODO: Write frontend-deployment.yaml and frontend-service.yaml
# Similar to the API but using .Values.frontend.* values

# Reference: Lesson 2.3, Lesson 3.2 (control flow)

cat > /tmp/taskly/templates/frontend-deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "taskly.frontendFullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
    app.kubernetes.io/component: frontend
spec:
  replicas: {{ .Values.frontend.replicaCount }}
  selector:
    matchLabels:
      {{- include "taskly.selectorLabels" . | nindent 6 }}
      app.kubernetes.io/component: frontend
  template:
    metadata:
      labels:
        {{- include "taskly.selectorLabels" . | nindent 8 }}
        app.kubernetes.io/component: frontend
    spec:
      containers:
        - name: frontend
          image: {{ .Values.frontend.image.repository }}:{{ .Values.frontend.image.tag }}
          imagePullPolicy: {{ .Values.frontend.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.frontend.service.port }}
EOF

cat > /tmp/taskly/templates/frontend-service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: {{ include "taskly.frontendFullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
    app.kubernetes.io/component: frontend
spec:
  type: {{ .Values.frontend.service.type }}
  ports:
    - port: {{ .Values.frontend.service.port }}
      targetPort: {{ .Values.frontend.service.port }}
      protocol: TCP
  selector:
    {{- include "taskly.selectorLabels" . | nindent 4 }}
    app.kubernetes.io/component: frontend
EOF

echo "Frontend templates written"

---

## Part 6 — Ingress

In [ ]:
# TODO: Write ingress.yaml that:
#   - Only renders if .Values.ingress.enabled is true
#   - Routes / to the frontend service
#   - Routes /api to the API service
#   - Conditionally includes TLS if .Values.ingress.tls is true

# Reference: Lesson 3.2 (control flow with if)

cat > /tmp/taskly/templates/ingress.yaml << 'EOF'
{{- if .Values.ingress.enabled }}
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: {{ include "taskly.fullname" . }}
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
  {{- with .Values.ingress.annotations }}
  annotations:
    {{- toYaml . | nindent 4 }}
  {{- end }}
spec:
  {{- if .Values.ingress.className }}
  ingressClassName: {{ .Values.ingress.className }}
  {{- end }}
  rules:
    - host: {{ .Values.ingress.host }}
      http:
        paths:
          - path: /api
            pathType: Prefix
            backend:
              service:
                name: {{ include "taskly.apiFullname" . }}
                port:
                  number: {{ .Values.api.service.port }}
          - path: /
            pathType: Prefix
            backend:
              service:
                name: {{ include "taskly.frontendFullname" . }}
                port:
                  number: {{ .Values.frontend.service.port }}
  {{- if .Values.ingress.tls }}
  tls:
    - hosts:
        - {{ .Values.ingress.host }}
      secretName: {{ include "taskly.fullname" . }}-tls
  {{- end }}
{{- end }}
EOF

echo "Ingress template written"

---

## Part 7 — Database Migration Hook

In [ ]:
# TODO: Write the pre-upgrade migration hook
# It should:
#   - Fire on pre-install AND pre-upgrade
#   - Be conditional on .Values.migration.enabled
#   - Run a Job that simulates DB schema migration
#   - Have hook-weight -5
#   - Use hook-delete-policy: before-hook-creation

# Reference: Lesson 4.3 (hooks)

cat > /tmp/taskly/templates/hooks/pre-upgrade-migrate.yaml << 'EOF'
{{- if .Values.migration.enabled }}
apiVersion: batch/v1
kind: Job
metadata:
  name: {{ .Release.Name }}-db-migrate
  namespace: {{ .Release.Namespace }}
  labels:
    {{- include "taskly.labels" . | nindent 4 }}
  annotations:
    "helm.sh/hook": pre-install,pre-upgrade
    "helm.sh/hook-weight": "-5"
    "helm.sh/hook-delete-policy": before-hook-creation
spec:
  template:
    spec:
      restartPolicy: Never
      initContainers:
        - name: wait-for-db
          image: busybox:1.36
          command:
            - /bin/sh
            - -c
            - |
              until nc -z {{ include "taskly.dbHost" . }} 5432; do
                echo "Waiting for PostgreSQL..."
                sleep 2
              done
              echo "PostgreSQL is ready!"
      containers:
        - name: migrate
          image: {{ .Values.migration.image.repository }}:{{ .Values.migration.image.tag }}
          command:
            - /bin/sh
            - -c
            - |
              echo "=== Taskly DB Migration ==="
              echo "Release: {{ .Release.Name }}"
              echo "App version: {{ .Chart.AppVersion }}"
              echo "Running migration against {{ include "taskly.dbHost" . }}:5432/{{ .Values.postgresql.auth.database }}"
              # In production: run your actual migration tool here
              # e.g.: alembic upgrade head
              # e.g.: python manage.py migrate
              echo "Migration complete!"
{{- end }}
EOF

echo "Migration hook written"

---

## Part 8 — Helm Tests

In [ ]:
# TODO: Write two test files:
#   1. test-api-connection.yaml — curl the API service endpoint
#   2. test-frontend-connection.yaml — wget the frontend service

# Reference: Lesson 5.2 (helm test)

cat > /tmp/taskly/templates/tests/test-api-connection.yaml << 'EOF'
apiVersion: v1
kind: Pod
metadata:
  name: {{ .Release.Name }}-test-api
  annotations:
    "helm.sh/hook": test
    "helm.sh/hook-delete-policy": hook-succeeded
spec:
  restartPolicy: Never
  containers:
    - name: test-api
      image: busybox:1.36
      command:
        - /bin/sh
        - -c
        - |
          API_HOST={{ include "taskly.apiFullname" . }}.{{ .Release.Namespace }}.svc.cluster.local
          echo "Testing API at ${API_HOST}:{{ .Values.api.service.port }}/health"
          wget --timeout=10 -qO- http://${API_HOST}:{{ .Values.api.service.port }}/ && echo "API test PASSED" || (echo "API test FAILED" && exit 1)
EOF

cat > /tmp/taskly/templates/tests/test-frontend-connection.yaml << 'EOF'
apiVersion: v1
kind: Pod
metadata:
  name: {{ .Release.Name }}-test-frontend
  annotations:
    "helm.sh/hook": test
    "helm.sh/hook-delete-policy": hook-succeeded
spec:
  restartPolicy: Never
  containers:
    - name: test-frontend
      image: busybox:1.36
      command:
        - /bin/sh
        - -c
        - |
          FRONTEND={{ include "taskly.frontendFullname" . }}.{{ .Release.Namespace }}.svc.cluster.local
          echo "Testing frontend at ${FRONTEND}:{{ .Values.frontend.service.port }}"
          wget --timeout=10 -qO- http://${FRONTEND}:{{ .Values.frontend.service.port }}/ | head -5
          echo "Frontend test PASSED!"
EOF

echo "Test templates written"

---

## Part 9 — NOTES.txt

In [ ]:
cat > /tmp/taskly/templates/NOTES.txt << 'EOF'
Thank you for installing {{ .Chart.Name }} {{ .Chart.AppVersion }}!

Release:   {{ .Release.Name }}
Namespace: {{ .Release.Namespace }}

Services deployed:
  API:      {{ include "taskly.apiFullname" . }}:{{ .Values.api.service.port }}
  Frontend: {{ include "taskly.frontendFullname" . }}:{{ .Values.frontend.service.port }}

{{- if .Values.ingress.enabled }}
Access the application at: http{{ if .Values.ingress.tls }}s{{ end }}://{{ .Values.ingress.host }}
{{- else }}
To access the frontend, run:
  kubectl port-forward svc/{{ include "taskly.frontendFullname" . }} 8080:{{ .Values.frontend.service.port }} -n {{ .Release.Namespace }}
  Then open: http://localhost:8080

To access the API:
  kubectl port-forward svc/{{ include "taskly.apiFullname" . }} 8000:{{ .Values.api.service.port }} -n {{ .Release.Namespace }}
  Then open: http://localhost:8000
{{- end }}

To run smoke tests:
  helm test {{ .Release.Name }} -n {{ .Release.Namespace }}

To upgrade:
  helm upgrade {{ .Release.Name }} <chart> -n {{ .Release.Namespace }} --reuse-values
EOF

echo "NOTES.txt written"

---

## Part 10 — Environment Overrides

In [ ]:
# Create environment-specific values files

cat > /tmp/taskly/values-dev.yaml << 'EOF'
# Development: minimal resources, single replicas
api:
  replicaCount: 1

frontend:
  replicaCount: 1

postgresql:
  primary:
    persistence:
      size: 500Mi

ingress:
  enabled: true
  host: taskly.dev.local
EOF

cat > /tmp/taskly/values-staging.yaml << 'EOF'
# Staging: moderate resources, ingress enabled
api:
  replicaCount: 2
  resources:
    limits:
      cpu: 500m
      memory: 256Mi

frontend:
  replicaCount: 2

ingress:
  enabled: true
  host: taskly.staging.example.com
EOF

cat > /tmp/taskly/values-production.yaml << 'EOF'
# Production: HA replicas, full resources, TLS ingress
api:
  replicaCount: 3
  resources:
    limits:
      cpu: 1000m
      memory: 512Mi
    requests:
      cpu: 200m
      memory: 256Mi

frontend:
  replicaCount: 3

ingress:
  enabled: true
  host: taskly.example.com
  tls: true
  annotations:
    cert-manager.io/cluster-issuer: letsencrypt-prod

postgresql:
  auth:
    # In production, reference a pre-created secret:
    existingSecret: taskly-db-credentials
    secretKeys:
      userPasswordKey: password
  primary:
    persistence:
      size: 20Gi
      storageClass: fast-ssd
EOF

echo "Environment values files created"

---

## Part 11 — Validate & Render

In [ ]:
# Download dependencies first
helm repo add bitnami https://charts.bitnami.com/bitnami 2>/dev/null || true
helm repo update

# TODO: Uncomment after completing Chart.yaml with the postgresql dependency
# helm dependency update /tmp/taskly

echo "=== Step 1: Lint the chart ==="
helm lint /tmp/taskly

echo ""
echo "=== Step 2: Render with dev values ==="
helm template taskly-dev /tmp/taskly \
  -f /tmp/taskly/values-dev.yaml \
  --namespace development \
  2>/dev/null | grep 'kind:' | sort | uniq -c

echo ""
echo "=== Step 3: Render with production values ==="
helm template taskly-prod /tmp/taskly \
  -f /tmp/taskly/values-production.yaml \
  --namespace production \
  --set api.image.tag=abc123 \
  --set frontend.image.tag=abc123 \
  2>/dev/null | grep 'kind:' | sort | uniq -c

In [ ]:
# Render the full production manifest and inspect key resources
helm template taskly-prod /tmp/taskly \
  -f /tmp/taskly/values-production.yaml \
  --namespace production \
  --set api.image.tag=v1.0.0 \
  --set frontend.image.tag=v1.0.0 \
  2>/dev/null | head -100

---

## Part 12 — Package & Publish (Simulation)

In [ ]:
# Package the chart
mkdir -p /tmp/taskly-dist
helm package /tmp/taskly --destination /tmp/taskly-dist

echo ""
ls -lh /tmp/taskly-dist/

echo ""
echo "To push to an OCI registry:"
echo 'helm registry login ghcr.io -u <username> --password-stdin'
echo 'helm push /tmp/taskly-dist/taskly-1.0.0.tgz oci://ghcr.io/<org>/charts'

---

## Part 13 — Live Deployment (Optional — requires a Kubernetes cluster)

In [ ]:
echo "=== Live deployment commands (requires cluster) ==="
echo ""
echo "# Install to dev namespace"
echo 'helm upgrade --install taskly /tmp/taskly \'
echo '  --namespace taskly-dev \'
echo '  --create-namespace \'
echo '  -f /tmp/taskly/values-dev.yaml \'
echo '  --wait --atomic'

echo ""
echo "# Run smoke tests"
echo 'helm test taskly --namespace taskly-dev'

echo ""
echo "# View release history"
echo 'helm history taskly --namespace taskly-dev'

echo ""
echo "# Simulate upgrade (bumping image tag)"
echo 'helm upgrade taskly /tmp/taskly \'
echo '  --namespace taskly-dev \'
echo '  -f /tmp/taskly/values-dev.yaml \'
echo '  --set api.image.tag=v1.0.1 \'
echo '  --wait --atomic'

echo ""
echo "# Rollback if needed"
echo 'helm rollback taskly --namespace taskly-dev'

echo ""
echo "# Cleanup"
echo 'helm uninstall taskly --namespace taskly-dev'

---

## Verification Checklist

Before declaring the capstone complete, verify each item:

- [ ] `helm lint /tmp/taskly` passes with 0 errors
- [ ] `helm template` renders both API and frontend Deployments
- [ ] `helm template` renders both Services
- [ ] Ingress is NOT rendered with default values (ingress.enabled=false)
- [ ] Ingress IS rendered when `ingress.enabled=true`
- [ ] TLS block appears only when `ingress.tls=true`
- [ ] Migration hook renders with `helm.sh/hook: pre-install,pre-upgrade`
- [ ] Migration hook does NOT render when `migration.enabled=false`
- [ ] Test pods render with `helm.sh/hook: test`
- [ ] Production values increase replica counts correctly
- [ ] `helm package` creates the `.tgz` successfully
- [ ] NOTES.txt renders with correct service names
- [ ] Standard labels (`app.kubernetes.io/*`) appear on all resources
- [ ] Selector labels do NOT contain the chart version

In [ ]:
# Run the checklist programmatically
CHART=/tmp/taskly

echo "=== Capstone Verification ==="
echo ""

# Check 1: Lint
helm lint ${CHART} > /dev/null 2>&1 && echo "[PASS] helm lint" || echo "[FAIL] helm lint"

# Check 2: API Deployment renders
helm template test ${CHART} 2>/dev/null | grep -q 'kind: Deployment' && echo "[PASS] Deployments render" || echo "[FAIL] No Deployments found"

# Check 3: Services render
helm template test ${CHART} 2>/dev/null | grep -q 'kind: Service' && echo "[PASS] Services render" || echo "[FAIL] No Services found"

# Check 4: Ingress not rendered by default
INGRESS_COUNT=$(helm template test ${CHART} 2>/dev/null | grep -c 'kind: Ingress' || echo 0)
[ "${INGRESS_COUNT}" = "0" ] && echo "[PASS] No Ingress with default values" || echo "[FAIL] Ingress rendered unexpectedly"

# Check 5: Ingress rendered when enabled
INGRESS_ENABLED=$(helm template test ${CHART} --set ingress.enabled=true 2>/dev/null | grep -c 'kind: Ingress' || echo 0)
[ "${INGRESS_ENABLED}" -gt "0" ] && echo "[PASS] Ingress renders when enabled" || echo "[FAIL] Ingress not rendering"

# Check 6: Migration hook renders
helm template test ${CHART} 2>/dev/null | grep -q 'helm.sh/hook.*pre-install' && echo "[PASS] Migration hook renders" || echo "[FAIL] Migration hook missing"

# Check 7: Labels present
helm template test ${CHART} 2>/dev/null | grep -q 'app.kubernetes.io/name' && echo "[PASS] Standard labels present" || echo "[FAIL] Standard labels missing"

echo ""
echo "Verification complete!"

---

## Stretch Goals

If you've completed the core project, try these extensions:

### Stretch Goal 1: Values Schema
Add `values.schema.json` that validates:
- `api.replicaCount` is an integer between 1 and 10
- `api.image.pullPolicy` is one of `Always`, `IfNotPresent`, `Never`
- `ingress.host` is a non-empty string when `ingress.enabled` is true

### Stretch Goal 2: Horizontal Pod Autoscaler
Add an optional HPA for both services, controlled by `api.autoscaling.enabled`.

### Stretch Goal 3: ServiceAccount & RBAC
Add a ServiceAccount for the API pod with controlled by `api.serviceAccount.create`.

### Stretch Goal 4: GitHub Actions Workflow
Write a `.github/workflows/helm.yml` that:
1. Lints on every PR
2. Packages and pushes to GHCR on merge to main
3. Deploys to a test cluster on tagged releases

### Stretch Goal 5: Flux HelmRelease
Write a `flux-helmrelease.yaml` that deploys the taskly chart to production namespace via Flux GitOps.

---

## Summary

Congratulations! You've built a complete, production-grade Helm chart that demonstrates:

| Skill | Where Applied |
|---|---|
| Chart structure | `taskly/` directory layout |
| Named templates | `_helpers.tpl` — fullname, labels, dbHost |
| Control flow | `ingress.yaml` — conditional TLS block |
| Dependencies | PostgreSQL subchart via bitnami |
| Hooks | `pre-upgrade-migrate.yaml` Job hook |
| Helm test | API + frontend connection tests |
| Multi-env values | dev / staging / production YAML overlays |
| Packaging | `helm package` + `helm push` |
| CI/CD | `helm upgrade --install --atomic` pattern |